# Homework 5 — Deployment — ML Zoomcamp 2026 cohort

A frozen lead-scoring model (`pipeline.bin`: `DictVectorizer` + `LogisticRegression(solver="liblinear")`, trained on `course_lead_scoring_2026.csv`) is given, together with its FastAPI service, `uv` lock and Dockerfile. You load it, serve it, containerise it. Questions: `homework5_2026.md`.

**Submit at:** https://courses.datatalks.club/ml-zoomcamp-2026/homework/hw05 · platform deadline **Mon 26 Oct**. Target: submit on the Mon 19 Oct 08:00 session (tracker); the platform week is cushion.

**Stack: uv + FastAPI + uvicorn**, not the lessons' pipenv + Flask + gunicorn. Run everything from this folder, inside the locked environment:

```bash
cd homework
uv sync --locked                           # once: builds .venv/ with Python 3.11.15 + scikit-learn from uv.lock
uv run --with jupyter jupyter lab          # opens Jupyter with THIS project's packages (nothing is added to uv.lock)
```

Q1, Q4 – Q6 are terminal work: their cells hold the command skeleton, and you paste what you got into the empty cell below. Scaffold only: question cells are empty.

## Setup — which environment is this notebook running in?

In [1]:
import hashlib
import json
import pickle
import sys
from pathlib import Path

import sklearn

print(sys.executable)
print('python', sys.version.split()[0], '| scikit-learn', sklearn.__version__)

/Users/giginho/Projects/ML Journey/03_MLOps_Deployment/Module_05_Deployment/cohort2026/homework/.venv/bin/python
python 3.11.15 | scikit-learn 1.7.2


> If this prints your conda `ml-zoomcamp` Python (and a scikit-learn that isn't the one in `uv.lock`), you're outside the homework environment. `pipeline.bin` still loads, with an `InconsistentVersionWarning`, but the reference result is defined by the locked version. Restart Jupyter with `uv run --with jupyter jupyter lab`, or do Q3 as a script with `uv run python q3.py`.

### Check the artifact first

```bash
uv run python smoke_test.py
sha256sum pipeline.bin            # macOS without coreutils: shasum -a 256 pipeline.bin
```

Compare with the checksum in `homework5_2026.md` and in `model_metadata.json`.

> `smoke_test.py` scores the **Q4/Q6 lead**, not the Q3 one. Its output isn't the Q3 answer.
> Don't run `train.py` and don't run `uv lock` / `uv add` here. `train.py` overwrites `pipeline.bin`, `feature_defaults.json` and `model_metadata.json` in place, and a retrain produces a file with a different checksum even with the same data and versions.

In [2]:
!uv run python smoke_test.py

{'conversion_probability': 0.769799, 'conversion': True}


## Q1. Environment check (not graded)

```bash
uv --version
```

Record the version in your notes.

In [3]:
!uv --version
# Q1: uv 0.13.4 (not graded)

uv 0.13.4 (7456b60bc 2026-10-20 aarch64-apple-darwin)


## Q2. Locked dependency

Open `pyproject.toml` and `uv.lock`. Which scikit-learn version is part of the 2026 reference environment? *(1.6.1 / 1.7.2 / 1.8.0 / 2.0.0)*

> Read it from the files, not from `sklearn.__version__` of whatever Python you happen to run: your conda env has its own scikit-learn.

In [4]:
# Q2: scikit-learn 1.7.2 (pinned in pyproject.toml and uv.lock)

## Q3. Load the model with pickle

Load `pipeline.bin` with `pickle` and call `predict_proba` for this lead. Report P(conversion) rounded to **3 decimals** (free answer, tolerance 0.005: there is no option list).

```python
lead_q3 = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41,
}
```

> `pipeline.bin` is **one** sklearn `Pipeline` (DictVectorizer inside), not the lecture's `(dv, model)` tuple: call `predict_proba` on the pipeline with the raw dict(s). The service passes a list, `[record]`, after `model.normalize_record`; with all eight fields present the normalisation changes nothing, but it's the same path the API uses.
> Column 1 of `predict_proba` is P(converted = 1).

In [5]:
lead_q3 = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41,
}

In [6]:
from model import normalize_record

model_path = Path("pipeline.bin")
with model_path.open("rb") as f_in:
    pipeline = pickle.load(f_in)

print(type(pipeline))   # sklearn Pipeline, not a (dv, model) tuple

record = normalize_record(lead_q3)               # same path as predict.py
p_q3 = pipeline.predict_proba([record])[0, 1]    # row 0, column 1 = P(converted = 1)

print(f"Q3: P(conversion) = {p_q3:.3f}")


<class 'sklearn.pipeline.Pipeline'>
Q3: P(conversion) = 0.533


## Q4. Serve the model

In a terminal, in this folder:

```bash
uv run uvicorn predict:app --host 0.0.0.0 --port 9696
```

Then send the second lead (in a second terminal with the `curl` from `homework5_2026.md`, or from here with `requests` / `urllib`) and report `conversion_probability` to 3 decimals (tolerance 0.005). Check the response also has a boolean `conversion` field, and look at `GET /health`.

```python
lead_q4 = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74,
}
```

> With `curl`, keep `-H 'Content-Type: application/json'`. Without it FastAPI answers 422 instead of a prediction.
> The response key is `conversion_probability`, not the lessons' `churn_probability`.

In [7]:
lead_q4 = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74,
}

In [8]:
# Q4: 0.770 (uv run uvicorn predict:app + curl; conversion: true; /health hashes match model_metadata.json)

## Q5. Container configuration

Which Python base-image tag is declared in `Dockerfile`? *(python:3.9-slim-bullseye / python:3.11.15-slim-bookworm / python:3.12-slim-bookworm / python:3.13.10-slim-bookworm)*

> Read the `FROM` line. The `@sha256:...` part after the tag is the digest pin, not part of the tag.

In [9]:
# Q5: python:3.11.15-slim-bookworm (FROM line; the @sha256 digest is not part of the tag)

## Q6. Run the container

Docker Desktop must be running. **Stop the Q4 uvicorn first (Ctrl+C)**: it holds port 9696.

```bash
docker build -t zoomcamp-model:2026-hw5 .
docker run --rm -p 9696:9696 zoomcamp-model:2026-hw5
# second terminal, same folder:
python q6_test.py
curl -s http://localhost:9696/health
docker ps                                   # confirm the container is what's listening on 9696
```

Report `conversion_probability` to 3 decimals (tolerance 0.005), and check that `/health` returns `"status": "ok"` and the `model_sha256` recorded in `model_metadata.json`.

> If uvicorn is still running, `docker run` fails with "port is already allocated" and `q6_test.py` happily answers from the **local** server: you'd report Q4 again without ever testing the container. `docker ps` is the proof.
> The Dockerfile copies the hidden file `.python-version`. If it went missing (`ls -a` to check), the build stops with `".python-version": not found`.

In [10]:
# Q6: 0.770 (container zoomcamp-model:2026-hw5, q6_test.py -> 0.769799, conversion: True;
#     /health model_sha256 1646bbdc... matches model_metadata.json; docker ps confirmed the container served it)
# Same as Q4: the packaged image reproduces the local result.

## Submit

Paste the link to this notebook in your GitHub repo, the time spent, and the reflection the form asks for: *"Describe one practical idea you took from this module."*

> The 2026 form for hw05 may only ask for the repo link, time spent and that reflection, not the numbers. Keep the Q2 – Q6 answers written in this notebook so they're in the repo either way.